# BGL LR — retain structured lists, including empty lists

This batch notebook processes extractive, abstractive and hybrid summary types for W=5,10,15. **Each row is one log chunk**, with `status_gt_label` used only as the supervised target/evaluation ground truth.

**Key change:** An explicitly provided list `[]` is a **valid summary representation**, not an invalid or missing summary. List keys, list elements, and empty-list structure are preserved as text features, using `EMPTY_LIST_TOKEN` for `[]` so the existing TF-IDF tokenization does not discard it. `None`, blank cells, and malformed JSON (when JSON is expected) are different from valid `[]`.

LR method unchanged: TF–IDF (1–2 grams), LogisticRegression, C grid, F1 selection on validation, default 0.5 decision threshold. Teacher train/validation (`GT`) and LogSumm student test predictions are distinct; verify label leakage before making scientific claims. Genuine missing/failed student outputs remain unpredicted and are reported in the output CSV.

# BGL — Batch Logistic Regression on LogSumm summaries

Run **all summary types × all chunk sizes** in one execution. Each row is a **chunk**, and the supervised target is the chunk-level `status_gt_label`.

**Algorithm unchanged from `BGL_LR_LogSumm_Student_Summaries.ipynb`:** TF–IDF (`ngram_range=(1,2)`, token pattern `\b\w+\b`) fitted only on training summaries; `LogisticRegression(max_iter=5000, random_state=42)`; `C = [0.01, 0.1, 1, 10, 100]`; choose best validation anomaly-F1; predict at the default decision threshold. No threshold tuning on test labels.

Train/validation default to the teacher-generated `GT` **summary**, while test defaults to the corresponding LogSumm **student-predicted summary** (not teacher `GT`). The configuration allows distinct columns per type and partition. Teacher → student is a domain shift and may involve label leakage if teacher summaries were built using event labels; consider training/validating on label-blind student predictions where possible.

**Files:** put all CSVs in one folder (or upload in Colab). Naming such as `BGL_split75-10-15_w5_seed42_train_unique_abstractive.csv` is recognized. A test file such as `BGL_W5~2.CSV` is recognized by its `student_abstractive_without_explanation` column. Missing combinations are recorded as skipped—never fabricated. All prediction rows (including missing student outputs) are preserved. Metrics are conditional on available predictions, and coverage is reported separately.

In [1]:
# 0 — Imports
# %pip install -q pandas numpy scikit-learn
import json, re, time, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support, f1_score, confusion_matrix, accuracy_score
from IPython.display import display


## 1 — Configuration: edit ONLY this cell

In [2]:
# =================== USER CONFIGURATION ===================
INPUT_DIR = Path('/content') if Path('/content').exists() else Path('/mnt/data')
OUTPUT_DIR = INPUT_DIR / 'bgl_lr_batch_results_fixed'

# Genuinely missing/failed training or validation summaries: 'drop' or 'error'
# Empty JSON lists [] are VALID and never dropped.
INVALID_TRAIN_POLICY = 'drop'
EMPTY_LIST_TOKEN = 'EMPTY_LIST'  # Represents the literal JSON [] in LR text features
# Test summaries are never silently dropped from prediction exports.


CHUNK_SIZES = [5, 10, 15]
SUMMARY_TYPES = ['extractive', 'abstractive', 'hybrid']

ID_COLUMN = 'segment_id'
LABEL_COLUMN = 'status_gt_label'
SPLIT_COLUMN = 'split'      # None if CSV files have no split column

# Training/validation columns may differ from the LogSumm test-prediction column.
TRAIN_COLUMNS = {t: 'GT' for t in SUMMARY_TYPES}
VALIDATION_COLUMNS = {t: 'GT' for t in SUMMARY_TYPES}
TEST_COLUMNS = {
    'extractive': 'student_extractive_without_explanation',
    'abstractive': 'student_abstractive_without_explanation',
    'hybrid': 'student_hybrid_without_explanation',
}
# For test files where *_without_explanation is absent, fall back to the raw student JSON.
TEST_FALLBACK_COLUMNS = {
    'extractive': ['student_extractive'],
    'abstractive': ['student_abstractive'],
    'hybrid': ['student_hybrid'],
}
TEST_STATUS_COLUMNS = {
    'extractive': 'student_extractive_status',
    'abstractive': 'student_abstractive_status',
    'hybrid': 'student_hybrid_status',
}
SUCCESS_STATUSES = {'completed', 'success', 'ok'}

# Summary parsing: 'auto' works for ordinary text and summary JSON.
TRAIN_FORMAT = 'auto'
VALIDATION_FORMAT = 'auto'
TEST_FORMAT = 'auto'

# Optional hard-coded test file overrides; use exact CSV names/paths when auto discovery is ambiguous.
# Example: (5,'abstractive'): 'BGL_W5~2.CSV'
TEST_FILE_OVERRIDES = {} # (5, 'abstractive'): 'BGL_W5~2.CSV'
TRAIN_FILE_OVERRIDES = {}         # (W, type): filename
VALIDATION_FILE_OVERRIDES = {}    # (W, type): filename

# Original previous summary-LR algorithm: DO NOT CHANGE to reproduce previous run.
C_GRID = [0.01, 0.1, 1, 10, 100]
SEED = 42
CLASS_WEIGHT = None
TFIDF_NGRAM_RANGE = (1, 2)

CHECK_SOURCE_ROW_OVERLAP = True
ON_INCOMPLETE = 'skip'   # 'skip' or 'raise'; never silently substitute GT for test predictions
DOWNLOAD_ZIP_IN_COLAB = True
# ===========================================================
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Input:', INPUT_DIR.resolve())
print('Output:', OUTPUT_DIR.resolve())


Input: /content
Output: /content/bgl_lr_batch_results_fixed


## 2 — Locate train / validation / student-prediction CSVs

In [3]:
def find_w(filename):
    match = re.search(r'(?i)(?:^|[_\-])w(5|10|15)(?!\d)', filename)
    return int(match.group(1)) if match else None

def partition_hint(name):
    name=name.lower()
    if re.search(r'(?:^|[_\-])train(?:[_\-.]|$)', name): return 'train'
    if re.search(r'(?:^|[_\-])validation(?:[_\-.]|$)', name) or re.search(r'(?:^|[_\-])val(?:[_\-.]|$)', name): return 'validation'
    if re.search(r'(?:^|[_\-])test(?:[_\-.]|$)', name): return 'test'
    return None

def column_names(file):
    return pd.read_csv(file, nrows=0).columns.tolist()

ALL_CSVS = sorted(INPUT_DIR.rglob('*.csv'))
# Exclude existing outputs to prevent accidentally selecting previous prediction files.
ALL_CSVS = [f for f in ALL_CSVS if OUTPUT_DIR not in f.parents]
print('Input CSV files found:', len(ALL_CSVS))

def select_file(W, typ, part, overrides):
    override=overrides.get((W,typ))
    if override:
        p=Path(override)
        p=p if p.is_absolute() else INPUT_DIR/p
        if not p.is_file(): raise FileNotFoundError(f'{part} override not found: {p}')
        return p

    candidates=[]
    for p in ALL_CSVS:
        if find_w(p.name)!=W: continue
        cols=set(column_names(p))
        hint=partition_hint(p.name)
        if part in ('train','validation'):
            # Use explicitly named files for teacher train/validation (not a student test file).
            if hint != part: continue
            if typ not in p.stem.lower(): continue
            if (TRAIN_COLUMNS if part=='train' else VALIDATION_COLUMNS)[typ] not in cols: continue
            rank=(0 if (part=='train' and 'unique' in p.stem.lower()) else 1, len(p.name))
        else:
            if hint in ('train','validation'): continue
            if not any(c in cols for c in [TEST_COLUMNS[typ]]+TEST_FALLBACK_COLUMNS[typ]): continue
            # Prefer test files explicitly tied to summary type, then student-only generic test files.
            rank=(0 if typ in p.stem.lower() else 1, 0 if hint=='test' else 1, len(p.name))
        candidates.append((rank,p))
    if not candidates:return None
    candidates.sort(key=lambda x:(x[0],str(x[1])))
    if len(candidates)>1 and candidates[0][0]==candidates[1][0]:
        raise ValueError(f'Ambiguous {part} for W={W} {typ}: {candidates[0][1]} and {candidates[1][1]}. Set a file override.')
    return candidates[0][1]

def pick_test_column(path,typ):
    cols=set(column_names(path))
    for col in [TEST_COLUMNS[typ]]+TEST_FALLBACK_COLUMNS[typ]:
        if col in cols: return col
    raise ValueError(f'No student prediction summary for type {typ} in {path.name}')

RUNS=[]
for w in CHUNK_SIZES:
    for typ in SUMMARY_TYPES:
        files={}
        for part,override in [('train',TRAIN_FILE_OVERRIDES),('validation',VALIDATION_FILE_OVERRIDES),('test',TEST_FILE_OVERRIDES)]:
            files[part]=select_file(w,typ,part,override)
        RUNS.append(dict(W=w,summary_type=typ,**files))
display(pd.DataFrame([{k: (v.name if isinstance(v,Path) else v) for k,v in run.items()} for run in RUNS]))


Input CSV files found: 31


,W,summary_type,train,validation,test
0,5,extractive,BGL_split75-10-15_w5_seed42_train_unique_extra...,BGL_split75-10-15_w5_seed42_validation_extract...,bgl_w5_extractive_stratified_test_predictions.csv
1,5,abstractive,BGL_split75-10-15_w5_seed42_train_unique_abstr...,BGL_split75-10-15_w5_seed42_validation_abstrac...,bgl_w5_abstractive_stratified_test_predictions...
2,5,hybrid,BGL_split75-10-15_w5_seed42_train_unique_hybri...,BGL_split75-10-15_w5_seed42_validation_hybrid.csv,bgl_w5_hybrid_stratified_test_predictions.csv
3,10,extractive,BGL_split75-10-15_w10_seed42_train_unique_extr...,BGL_split75-10-15_w10_seed42_validation_extrac...,bgl_w10_extractive_stratified_test_predictions...
4,10,abstractive,BGL_split75-10-15_w10_seed42_train_unique_abst...,BGL_split75-10-15_w10_seed42_validation_abstra...,bgl_w10_abstractive_stratified_test_prediction...
5,10,hybrid,BGL_split75-10-15_w10_seed42_train_unique_hybr...,BGL_split75-10-15_w10_seed42_validation_hybrid...,bgl_w10_hybrid_stratified_test_predictions.csv
6,15,extractive,BGL_split75-10-15_w15_seed42_train_unique_extr...,BGL_split75-10-15_w15_seed42_validation_extrac...,bgl_w15_extractive_stratified_test_predictions...
7,15,abstractive,BGL_split75-10-15_w15_seed42_train_unique_abst...,BGL_split75-10-15_w15_seed42_validation_abstra...,bgl_w15_abstractive_stratified_test_prediction...
8,15,hybrid,BGL_split75-10-15_w15_seed42_train_unique_hybr...,BGL_split75-10-15_w15_seed42_validation_hybrid...,bgl_w15_hybrid_stratified_test_predictions.csv


## 3 — Summary parsing, label validation, and overlap checks (reused from previous notebook)

In [4]:
EXCLUDED_KEYS = {
    'explanation','explanations','reasoning','rationale','chain_of_thought',
    'status_gt_label','event_labels_json','label','labels','ground_truth','ground_truth_label',
    'gt_label','anomalous_event_count','raw_response','content_with_labels'
}
PREFERRED_KEYS = ['summary','critical_events','context_events','activities','normal_events',
                  'selected_events','events','faults','analysis','diagnosis','conclusion']

# Keep list structure and keys; [] is a valid, intentionally empty list. A
# TF-IDF vectorizer using token_pattern=r'(?u)\b\w+\b' ignores bare brackets,
# so write a consistent feature token instead of treating [] as empty text.
def render_summary_object(obj):
    if obj is None:
        return ''
    if isinstance(obj, str):
        return obj.strip()
    if isinstance(obj, list):
        if not obj:
            return EMPTY_LIST_TOKEN
        return ' list_item '.join(render_summary_object(v) or 'EMPTY_ITEM' for v in obj)
    if isinstance(obj, dict):
        allowed = {str(k): v for k,v in obj.items()
                   if str(k).strip().lower() not in EXCLUDED_KEYS}
        if 'summary' in allowed and isinstance(allowed['summary'], str):
            # For standard abstractive JSON keep only the actual summary text.
            return allowed['summary'].strip()
        keys = [k for k in PREFERRED_KEYS if k in allowed]
        keys += [k for k in allowed if k not in keys]
        pieces=[]
        for k in keys:
            v=allowed[k]
            val=render_summary_object(v)
            # Preserve empty lists as `critical_events: EMPTY_LIST`, etc.
            if val:
                pieces.append(f'{k}: {val}')
        return '\n'.join(pieces)
    return str(obj)

def normalize_summary(cell, fmt='auto'):
    if cell is None or (not isinstance(cell, (dict,list)) and pd.isna(cell)):
        return ''
    if isinstance(cell,(dict,list)):
        obj=cell
    else:
        s=str(cell).strip()
        if not s:
            return ''
        if fmt=='plain':
            return s
        try:
            obj=json.loads(s)
        except (json.JSONDecodeError,TypeError):
            if fmt in ('json_summary','json_fields'):
                raise ValueError('Requested JSON summary format but value is not valid JSON')
            # Non-JSON plain-text summaries are legitimate, too.
            return s
    if fmt=='json_summary':
        if not isinstance(obj,dict) or 'summary' not in obj:
            raise ValueError('JSON object missing `summary` field')
        return render_summary_object(obj['summary'])
    return render_summary_object(obj)

LABEL_ANNOTATION_RE=re.compile(r'\[\s*label\s*=',re.I)

def load_split(path, split_name, summary_col, fmt, status_col=None):
    d=pd.read_csv(path,low_memory=False)
    needed=[ID_COLUMN,LABEL_COLUMN,summary_col]
    if SPLIT_COLUMN is not None:needed += [SPLIT_COLUMN]
    if status_col is not None:needed += [status_col]
    missing=[c for c in needed if c not in d.columns]
    if missing:raise ValueError(f'{path.name}: missing columns {missing}; available: {d.columns.tolist()}')
    if d.empty:raise ValueError(f'{path.name}: empty CSV')
    if d[ID_COLUMN].isna().any() or d[ID_COLUMN].astype(str).duplicated().any():
        raise ValueError(f'{path.name}: chunk IDs missing or duplicated')
    if SPLIT_COLUMN:
        split_aliases={'train':{'train','training'},'validation':{'validation','val','valid','dev'},'test':{'test','testing'}}
        observed=d[SPLIT_COLUMN].astype(str).str.strip().str.lower()
        if not observed.isin(split_aliases[split_name]).all():
            raise ValueError(f'{path.name}: unexpected {SPLIT_COLUMN} values: {sorted(observed.unique())}')
    labels=d[LABEL_COLUMN].astype(str).str.strip().str.lower()
    if not labels.isin(['normal','anomalous']).all():
        raise ValueError(f'{path.name}: invalid chunk labels: {sorted(labels.unique())}')
    d['_y']=labels.eq('anomalous').astype(int)
    if 'event_count' in d:
        count=pd.to_numeric(d['event_count'],errors='coerce')
        if count.isna().any() or (count != CHUNK_SIZE).any():
            raise ValueError(f'{path.name}: event_count differs from configured W={CHUNK_SIZE}')
    d['_summary']=d[summary_col].map(lambda v: normalize_summary(v,fmt))
    if d['_summary'].str.contains(LABEL_ANNOTATION_RE).any():
        raise ValueError(f'{path.name}: summary includes [label=...] annotation')
    # A valid [] produces the non-empty `EMPTY_LIST` token, while a genuinely
    # missing field produces ''. This does not inspect the ground-truth label.
    d['_valid_summary']=d['_summary'].str.strip().ne('')
    if status_col:
        d['_valid_summary'] &= d[status_col].astype(str).str.strip().str.lower().isin(SUCCESS_STATUSES)
    if split_name in ('train','validation') and not d['_valid_summary'].all():
        nbad=int((~d['_valid_summary']).sum())
        print(f'WARNING {path.name}: {nbad}/{len(d)} genuinely missing/failed {split_name} summaries')
        print('Class counts (diagnostics only):',d.loc[~d['_valid_summary'],LABEL_COLUMN].value_counts().to_dict())
    return d


In [5]:
# Sanity check: list structure is retained and empty lists are valid LR inputs.
assert normalize_summary('[]') == EMPTY_LIST_TOKEN
assert normalize_summary('{"critical_events": [], "context_events": []}') == (
    'critical_events: EMPTY_LIST\ncontext_events: EMPTY_LIST')
assert normalize_summary('{"critical_events": ["Disk failure"], "context_events": []}') == (
    'critical_events: Disk failure\ncontext_events: EMPTY_LIST')
assert normalize_summary('') == ''
print('Summary structure checks passed: [] is valid; missing text stays missing.')

Summary structure checks passed: [] is valid; missing text stays missing.


In [6]:
def check_disjoint(tr, va, te):
    for a_name,a,b_name,b in [('train',tr,'validation',va),('train',tr,'test',te),('validation',va,'test',te)]:
        ids_a=set(a[ID_COLUMN].astype(str));ids_b=set(b[ID_COLUMN].astype(str))
        overlap=ids_a&ids_b
        if overlap:raise ValueError(f'Overlapping segment IDs in {a_name}/{b_name}: {len(overlap)}')
        if CHECK_SOURCE_ROW_OVERLAP and 'source_rows_json' in a and 'source_rows_json' in b:
            rows_a={r for cell in a['source_rows_json'].dropna() for r in json.loads(cell)}
            rows_b={r for cell in b['source_rows_json'].dropna() for r in json.loads(cell)}
            overlap_rows=rows_a&rows_b
            if overlap_rows:raise ValueError(f'Overlapping log event IDs in {a_name}/{b_name}: {len(overlap_rows)}')
    if tr['_y'].nunique()<2 or va['_y'].nunique()<2:
        raise ValueError('Training and validation partitions must contain both normal and anomalous chunks')


## 4 — Run all available experiments; preserve the LR algorithm

In [7]:
all_metrics=[]
all_predictions=[]
run_inventory=[]

for run in RUNS:
    CHUNK_SIZE=run['W']  # load_split verifies event_count against this W
    typ=run['summary_type']
    tag=f'w{CHUNK_SIZE}_{typ}'
    missing=[p for p in ('train','validation','test') if run[p] is None]
    if missing:
        reason='Missing '+', '.join(missing)+' CSV'
        run_inventory.append(dict(W=CHUNK_SIZE,summary_type=typ,status='skipped',reason=reason))
        if ON_INCOMPLETE=='raise':raise FileNotFoundError(f'{tag}: {reason}')
        print(f'SKIP {tag}: {reason}')
        continue
    print(f'\n=== {tag} ===')
    test_col=pick_test_column(run['test'],typ)
    student_status_col=TEST_STATUS_COLUMNS.get(typ)
    if student_status_col not in column_names(run['test']):student_status_col=None

    tr=load_split(run['train'],'train',TRAIN_COLUMNS[typ],TRAIN_FORMAT)
    va=load_split(run['validation'],'validation',VALIDATION_COLUMNS[typ],VALIDATION_FORMAT)
    te=load_split(run['test'],'test',test_col,TEST_FORMAT,student_status_col)
    # Verify partition isolation on ALL rows, before any filtering.
    check_disjoint(tr,va,te)

    n_train_original, n_validation_original = len(tr), len(va)
    n_train_invalid = int((~tr['_valid_summary']).sum())
    n_validation_invalid = int((~va['_valid_summary']).sum())
    if n_train_invalid or n_validation_invalid:
        if INVALID_TRAIN_POLICY == 'error':
            raise ValueError(f'{tag}: invalid training/validation summaries: '
                             f'train={n_train_invalid}, validation={n_validation_invalid}')
        if INVALID_TRAIN_POLICY != 'drop':
            raise ValueError(f'Unknown INVALID_TRAIN_POLICY: {INVALID_TRAIN_POLICY}')
        # Use only summary text that actually exists. Never replace a missing
        # training summary with a ground-truth-derived normal/anomalous sentence.
        tr = tr.loc[tr['_valid_summary']].copy()
        va = va.loc[va['_valid_summary']].copy()
        print(f'{tag}: excluded {n_train_invalid} training and '
              f'{n_validation_invalid} validation chunks with missing/failed summaries')
    if tr.empty or va.empty or tr['_y'].nunique() < 2 or va['_y'].nunique() < 2:
        reason = (f'Insufficient valid training/validation data after filtering: '
                  f'train={len(tr)}, classes={tr["_y"].nunique()}, '
                  f'validation={len(va)}, classes={va["_y"].nunique()}')
        print(f'SKIP {tag}: {reason}')
        run_inventory.append(dict(W=CHUNK_SIZE, summary_type=typ,
                                  status='skipped', reason=reason,
                                  n_train_invalid=n_train_invalid,
                                  n_validation_invalid=n_validation_invalid))
        continue

    # UNCHANGED LR ALGORITHM FROM PREVIOUS NOTEBOOK
    vectorizer = TfidfVectorizer(lowercase=True, token_pattern=r'(?u)\b\w+\b', ngram_range=TFIDF_NGRAM_RANGE)
    Xtr=vectorizer.fit_transform(tr['_summary'])
    Xva=vectorizer.transform(va['_summary'])
    ytr=tr['_y'].to_numpy(); yva=va['_y'].to_numpy()
    best=None
    for C in C_GRID:
        clf=LogisticRegression(C=C,class_weight=CLASS_WEIGHT,max_iter=5000,random_state=SEED)
        clf.fit(Xtr,ytr)
        val_f1=f1_score(yva,clf.predict(Xva),zero_division=0)
        if best is None or val_f1>best[0]: best=(val_f1,C,clf)
    val_f1,best_C,model=best
    # END UNCHANGED LR TRAINING AND HYPERPARAMETER SELECTION

    valid_mask=te['_valid_summary'].to_numpy(dtype=bool)
    pred=np.full(len(te),np.nan)
    prob=np.full(len(te),np.nan)
    t0=time.perf_counter()
    if valid_mask.any():
        Xte=vectorizer.transform(te.loc[valid_mask,'_summary'])
        pred[valid_mask]=model.predict(Xte).astype(float)
        prob[valid_mask]=model.predict_proba(Xte)[:,list(model.classes_).index(1)]
    predict_seconds=time.perf_counter()-t0

    out=pd.DataFrame({
        'segment_id':te[ID_COLUMN].astype(str).to_numpy(),
        'W':CHUNK_SIZE, 'summary_type':typ,
        'test_source_column':test_col,
        'logsumm_summary_text':te['_summary'].to_numpy(),
        'status_gt_label':te[LABEL_COLUMN].to_numpy(),
        'ground_truth_binary':te['_y'].to_numpy(),
        'lr_prediction_binary':pd.array(pred,dtype='Int64'),
        'lr_prediction':np.where(~np.isnan(pred),np.where(pred==1,'anomalous','normal'),''),
        'lr_anomaly_probability':prob,
        'prediction_status':np.where(valid_mask,'predicted','missing_or_failed_summary'),
    })
    if student_status_col:
        out['logsumm_generation_status']=te[student_status_col].to_numpy()
    prediction_path=OUTPUT_DIR/f'BGL_LR_{tag}_chunk_predictions.csv'
    out.to_csv(prediction_path,index=False)
    all_predictions.append(out)

    n_total=len(te); n_valid=int(valid_mask.sum()); n_failed=n_total-n_valid
    if n_valid:
        y=te.loc[valid_mask,'_y'].to_numpy(dtype=int)
        p=pred[valid_mask].astype(int)
        P,R,F,_=precision_recall_fscore_support(y,p,average='binary',zero_division=0)
        tn,fp,fn,tp=confusion_matrix(y,p,labels=[0,1]).ravel()
        accuracy=accuracy_score(y,p)
    else:
        P=R=F=accuracy=np.nan
        tn=fp=fn=tp=0
    metrics={
        'W':CHUNK_SIZE,'summary_type':typ,'train_source_column':TRAIN_COLUMNS[typ],
        'validation_source_column':VALIDATION_COLUMNS[typ],'test_source_column':test_col,
        'n_train':len(tr),'n_validation':len(va),
        'n_train_original':n_train_original,'n_validation_original':n_validation_original,
        'n_train_invalid':n_train_invalid,'n_validation_invalid':n_validation_invalid,
        'n_test_total':n_total,
        'n_test_predicted':n_valid,'n_test_missing_or_failed':n_failed,
        'test_coverage':n_valid/n_total,'anomalous_test_total':int(te['_y'].sum()),
        'best_C':best_C,'validation_F1':val_f1,'precision':P,'recall':R,'F1':F,
        'accuracy':accuracy,'TP':int(tp),'FP':int(fp),'TN':int(tn),'FN':int(fn),
        'n_features':len(vectorizer.vocabulary_),'test_inference_seconds':predict_seconds,
        'prediction_csv':prediction_path.name,
        'train_file':run['train'].name,'validation_file':run['validation'].name,'test_file':run['test'].name
    }
    all_metrics.append(metrics)
    run_inventory.append(dict(W=CHUNK_SIZE,summary_type=typ,status='completed',reason='',
                              n_train_invalid=n_train_invalid,n_validation_invalid=n_validation_invalid))
    print(f"{tag}: P={P:.4f} R={R:.4f} F1={F:.4f}; {n_valid}/{n_total} predictions; C={best_C}")

metrics_df=pd.DataFrame(all_metrics)
inventory_df=pd.DataFrame(run_inventory)
metrics_path=OUTPUT_DIR/'BGL_LR_ALL_TYPES_ALL_CHUNKS_metrics.csv'
status_path=OUTPUT_DIR/'BGL_LR_run_status.csv'
metrics_df.to_csv(metrics_path,index=False)
inventory_df.to_csv(status_path,index=False)

predictions_path=OUTPUT_DIR/'BGL_LR_ALL_TYPES_ALL_CHUNKS_predictions.csv'
if all_predictions:
    pd.concat(all_predictions,ignore_index=True).to_csv(predictions_path,index=False)
else:
    pd.DataFrame().to_csv(predictions_path,index=False)

print('\n=== Final combined metrics ===')
display(metrics_df)
print('\n=== Run status (includes skipped combinations) ===')
display(inventory_df)
print('Saved:',metrics_path, predictions_path, status_path,sep='\n')



=== w5_extractive ===
w5_extractive: P=0.9589 R=0.9333 F1=0.9459; 1073/1077 predictions; C=10

=== w5_abstractive ===
w5_abstractive: P=0.8500 R=0.8831 F1=0.8662; 1077/1077 predictions; C=10

=== w5_hybrid ===
w5_hybrid: P=0.9667 R=0.7632 F1=0.8529; 1074/1077 predictions; C=1

=== w10_extractive ===
w10_extractive: P=0.8641 R=0.9780 F1=0.9175; 1298/1304 predictions; C=1

=== w10_abstractive ===
w10_abstractive: P=0.9059 R=0.8191 F1=0.8603; 1304/1304 predictions; C=100

=== w10_hybrid ===
w10_hybrid: P=0.9851 R=0.7416 F1=0.8462; 1297/1304 predictions; C=1

=== w15_extractive ===
w15_extractive: P=1.0000 R=0.5727 F1=0.7283; 1299/1302 predictions; C=100

=== w15_abstractive ===
w15_abstractive: P=0.9053 R=0.7748 F1=0.8350; 1302/1302 predictions; C=1

=== w15_hybrid ===
w15_hybrid: P=0.8947 R=0.6182 F1=0.7312; 1300/1302 predictions; C=1

=== Final combined metrics ===


,W,summary_type,train_source_column,validation_source_column,test_source_column,n_train,n_validation,n_train_original,n_validation_original,n_train_invalid,...,TP,FP,TN,FN,n_features,test_inference_seconds,prediction_csv,train_file,validation_file,test_file
0,5,extractive,GT,GT,student_extractive_without_explanation,605,156,605,156,0,...,70,3,995,5,3535,0.026619,BGL_LR_w5_extractive_chunk_predictions.csv,BGL_split75-10-15_w5_seed42_train_unique_extra...,BGL_split75-10-15_w5_seed42_validation_extract...,bgl_w5_extractive_stratified_test_predictions.csv
1,5,abstractive,GT,GT,student_abstractive_without_explanation,605,156,605,156,0,...,68,12,988,9,1431,0.021087,BGL_LR_w5_abstractive_chunk_predictions.csv,BGL_split75-10-15_w5_seed42_train_unique_abstr...,BGL_split75-10-15_w5_seed42_validation_abstrac...,bgl_w5_abstractive_stratified_test_predictions...
2,5,hybrid,GT,GT,student_hybrid_without_explanation,605,156,605,156,0,...,58,2,996,18,3480,0.042696,BGL_LR_w5_hybrid_chunk_predictions.csv,BGL_split75-10-15_w5_seed42_train_unique_hybri...,BGL_split75-10-15_w5_seed42_validation_hybrid.csv,bgl_w5_hybrid_stratified_test_predictions.csv
3,10,extractive,GT,GT,student_extractive_without_explanation,535,165,535,165,0,...,89,14,1193,2,3734,0.033110,BGL_LR_w10_extractive_chunk_predictions.csv,BGL_split75-10-15_w10_seed42_train_unique_extr...,BGL_split75-10-15_w10_seed42_validation_extrac...,bgl_w10_extractive_stratified_test_predictions...
4,10,abstractive,GT,GT,student_abstractive_without_explanation,535,165,535,165,0,...,77,8,1202,17,1353,0.027886,BGL_LR_w10_abstractive_chunk_predictions.csv,BGL_split75-10-15_w10_seed42_train_unique_abst...,BGL_split75-10-15_w10_seed42_validation_abstra...,bgl_w10_abstractive_stratified_test_prediction...
5,10,hybrid,GT,GT,student_hybrid_without_explanation,535,165,535,165,0,...,66,1,1207,23,3684,0.050981,BGL_LR_w10_hybrid_chunk_predictions.csv,BGL_split75-10-15_w10_seed42_train_unique_hybr...,BGL_split75-10-15_w10_seed42_validation_hybrid...,bgl_w10_hybrid_stratified_test_predictions.csv
6,15,extractive,GT,GT,student_extractive_without_explanation,509,142,509,142,0,...,63,0,1189,47,3970,0.029398,BGL_LR_w15_extractive_chunk_predictions.csv,BGL_split75-10-15_w15_seed42_train_unique_extr...,BGL_split75-10-15_w15_seed42_validation_extrac...,bgl_w15_extractive_stratified_test_predictions...
7,15,abstractive,GT,GT,student_abstractive_without_explanation,509,142,509,142,0,...,86,9,1182,25,1546,0.024849,BGL_LR_w15_abstractive_chunk_predictions.csv,BGL_split75-10-15_w15_seed42_train_unique_abst...,BGL_split75-10-15_w15_seed42_validation_abstra...,bgl_w15_abstractive_stratified_test_prediction...
8,15,hybrid,GT,GT,student_hybrid_without_explanation,509,142,509,142,0,...,68,8,1182,42,3619,0.048508,BGL_LR_w15_hybrid_chunk_predictions.csv,BGL_split75-10-15_w15_seed42_train_unique_hybr...,BGL_split75-10-15_w15_seed42_validation_hybrid...,bgl_w15_hybrid_stratified_test_predictions.csv



=== Run status (includes skipped combinations) ===


,W,summary_type,status,reason,n_train_invalid,n_validation_invalid
0,5,extractive,completed,,0,0
1,5,abstractive,completed,,0,0
2,5,hybrid,completed,,0,0
3,10,extractive,completed,,0,0
4,10,abstractive,completed,,0,0
5,10,hybrid,completed,,0,0
6,15,extractive,completed,,0,0
7,15,abstractive,completed,,0,0
8,15,hybrid,completed,,0,0


Saved:
/content/bgl_lr_batch_results_fixed/BGL_LR_ALL_TYPES_ALL_CHUNKS_metrics.csv
/content/bgl_lr_batch_results_fixed/BGL_LR_ALL_TYPES_ALL_CHUNKS_predictions.csv
/content/bgl_lr_batch_results_fixed/BGL_LR_run_status.csv


## 5 — Download ONE ZIP with combined metrics, combined predictions, individual predictions and run status

In [8]:
zip_path=OUTPUT_DIR/'BGL_LR_ALL_TYPES_ALL_CHUNKS_results.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for path in [metrics_path,predictions_path,status_path]+sorted(OUTPUT_DIR.glob('BGL_LR_w*_chunk_predictions.csv')):
        if path.exists(): z.write(path,arcname=path.name)
print('Created:',zip_path)
try:
    import google.colab
    if DOWNLOAD_ZIP_IN_COLAB:
        from google.colab import files
        files.download(str(zip_path))
except ImportError:
    print('Local Jupyter: files are located in',OUTPUT_DIR.resolve())


Created: /content/bgl_lr_batch_results_fixed/BGL_LR_ALL_TYPES_ALL_CHUNKS_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>